## Setup

In [44]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np
import threading
import asyncio
import matplotlib.pyplot as plt

LOG.propagate = False
# Get ArtemisBLEController object
ble = get_ble_controller()

def connectBLE(ble, timeout_s=10.0):
    # Fast path: already connected
    try:
        if ble.is_connected():
            return True
    except Exception:
        pass

    # Reload config ONCE (important)
    ble.reload_config()

    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            ble.connect()
        except Exception as e:
            # If it says already connected, treat as success
            if "already connected" in str(e).lower():
                return True

        # Give the stack a moment, then re-check
        time.sleep(0.2)
        try:
            if ble.is_connected():
                return True
        except Exception:
            pass

        time.sleep(0.3)

    raise TimeoutError("Failed to connect to BLE device")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Connect

In [46]:
connectBLE(ble)

2026-02-10 20:08:46,880 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:42:d1:79:ab:49
2026-02-10 20:08:46,880 | INFO     |: Scanning for device with address: c0:42:d1:79:ab:49, service UUID: 4843fd62-068c-4a37-af34-e724c6a05681
2026-02-10 20:08:57,002 | INFO     |: Found 18 total devices
2026-02-10 20:08:57,003 | INFO     |: Found matching device: C0:42:D1:79:AB:49 (name: Artemis BLE)
2026-02-10 20:08:58,229 | INFO     |: Connected to c0:42:d1:79:ab:49


True

## Disconnect

In [ ]:
ble.disconnect()

## Echo

In [ ]:
# Send the message
ble.send_command(CMD.ECHO, "Hello World!")

# Check that the board received it and returns it
s = ble.receive_string(ble.uuid['RX_STRING'])
print("Recieved from board: " + s)

## Send Three Floats

In [ ]:
ble.send_command(CMD.SEND_THREE_FLOATS, "3.14|5.25|7.432")

## Get Time Millis

In [ ]:
# Send command
ble.send_command(CMD.GET_TIME_MILLIS, "")

# Check board message
s = ble.receive_string(ble.uuid['RX_STRING'])
print("Recieved from board: " + s)

## Notification Handler

In [ ]:
def notification_callback(uuid, charac_bytearray):
    msg = ble.bytearray_to_string(charac_bytearray)
    if msg.startswith("T:"):
        t_ms = int(msg.split(":")[1])
        print("Time (ms):", t_ms)
    else:
        print("Unexpected message:", msg)

duration_s = 0.3  # set collection time

# Set up callback
ble.start_notify(ble.uuid['RX_STRING'], notification_callback)

t0 = time.time()
while (time.time() - t0) < duration_s:
    time.sleep(0.01)   # small sleep so you don't busy-wait

ble.stop_notify(ble.uuid['RX_STRING'])
print("End time notify")




## Time Array

In [ ]:
done = threading.Event()
time_values = []

def time_array_notification_callback(uuid, data: bytearray):
    msg = ble.bytearray_to_string(data).strip()
    print("Msg received:", repr(msg))

    parts = [p.strip() for p in msg.split(",") if p.strip()]
    if parts and parts[-1].lower() == "end":
        time_values.extend(parts[:-1])
        print("End of values")
        done.set()
        return
    time_values.extend(parts)

def get_time_data(timeout_s=10.0):
    time_values.clear()
    done.clear()
    
    # send command (NO await)
    ble.send_command(CMD.SEND_TIME_DATA, "")

    t0 = time.time()
    while not done.is_set() and (time.time() - t0) < timeout_s:
        ble.sleep(0.01)

    if not done.is_set():
        print("Timeout reached.")
    return list(time_values)

# start notify
try:
    ble.start_notify(ble.uuid['RX_STRING'], time_array_notification_callback)
except Exception as e:
    if "Notify acquired" in str(e):
        print("Notify already active; continuing.")
time.sleep(0.2)

vals = get_time_data()
print(vals)

try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception as e:
    print("Failed to stop notifications with exception: ", e)


## Temp Readings

In [ ]:
done = threading.Event()
time_values = []
temp_values = []

def temp_array_notification_callback(uuid, data: bytearray):
    msg = ble.bytearray_to_string(data).strip()
    print("Msg received:", repr(msg))

    parts = [p.strip() for p in msg.split(",") if p.strip()]
    for token in parts:
        if token.lower() == "end":
            print("End of values")
            done.set()
            return

        try:
            time_str, t_str = token.split(":", 1)
            temp_values.append(int(t_str))
            time_values.append(int(time_str))
        except ValueError:
            print("Bad token:", repr(token))

def get_temp_data(timeout_s=10.0):
    time_values.clear()
    temp_values.clear()
    done.clear()
    
    # send command (NO await)
    ble.send_command(CMD.GET_TEMP_READINGS, "")

    t0 = time.time()
    while not done.is_set() and (time.time() - t0) < timeout_s:
        ble.sleep(0.01)

    if not done.is_set():
        print("Timeout reached.")
    return list(temp_values), list(time_values)

# start notify
try:
    ble.start_notify(ble.uuid['RX_STRING'], temp_array_notification_callback)
except Exception as e:
    if "Notify acquired" in str(e):
        print("Notify already active; continuing.")
time.sleep(0.2)

temp_vals, time_vals = get_temp_data()

print("Temp array length: ", len(temp_values))
print("Time array length: ", len(time_values))
print("Temp values: ",temp_vals)
print("Time values:", time_vals)

try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception as e:
    print("Failed to stop notifications with exception: ", e)


## Data Rate

In [ ]:
# Code generated by AI
msg_sizes = [5, 10, 25, 50, 100, 120]
transmit_times = {}
received_data = []

def data_rate_callback(uuid, data: bytearray):
    msg = ble.bytearray_to_string(data).strip()
    rx_time = time.perf_counter()
    size = len(msg)
    received_data.append((rx_time, size))
    print(f"Callback: len={size}")

# Start notify
try:
    ble.start_notify(ble.uuid['RX_STRING'], data_rate_callback)
except Exception as e:
    if "Notify acquired" in str(e):
        print("Notify already active; continuing.")

time.sleep(1.0)
received_data.clear()

# Send all requests, recording transmit times by size
for size in msg_sizes:
    print(f"Sending request for {size} bytes...")
    transmit_times[size] = time.perf_counter()
    ble.send_command(CMD.DATA_RATE, str(size))
    time.sleep(0.1)  # Short delay between sends

# Send a dummy request immediately to flush the last real response
print("Sending dummy request to flush...")
ble.send_command(CMD.DATA_RATE, "1")
time.sleep(1.0)  # Wait for all responses

# Stop notifications
try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception as e:
    print("Failed to stop notifications")

# Debug output
print(f"\nReceived {len(received_data)} responses")
for rx_time, size in received_data:
    print(f"  Size: {size}")

# Match received responses to transmit times by size
# Filter out the dummy response (size 1)
results = []
for rx_time, size in received_data:
    if size in transmit_times and size in msg_sizes:
        rtt = rx_time - transmit_times[size]
        results.append((size, rtt))

# Sort by size
results.sort(key=lambda x: x[0])

# Display results
print("\n--- Data Rate Results ---")
actual_sizes = []
round_trip_times = []
data_rates = []

for size, rtt in results:
    one_way = rtt / 2
    rate = size / one_way
    actual_sizes.append(size)
    round_trip_times.append(rtt * 1000)
    data_rates.append(rate)
    print(f"{size:3d} bytes: RTT = {rtt*1000:.2f} ms, "
          f"One-way = {one_way*1000:.2f} ms, "
          f"Data rate = {rate:.2f} bytes/sec")

# Plot
if len(results) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    ax1.plot(actual_sizes, round_trip_times, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Message Size (bytes)')
    ax1.set_ylabel('Round-Trip Time (ms)')
    ax1.set_title('Round-Trip Time vs Message Size')
    ax1.grid(True)

    ax2.plot(actual_sizes, data_rates, 'ro-', linewidth=2, markersize=8)
    ax2.set_xlabel('Message Size (bytes)')
    ax2.set_ylabel('Data Rate (bytes/sec)')
    ax2.set_title('Data Rate vs Message Size')
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig('ble_data_rate.png')
    plt.show()

### IMU Data

In [50]:
done = threading.Event()
time_values = []
imu_values = []

def imu_array_notification_callback(uuid, data: bytearray):
    msg = ble.bytearray_to_string(data).strip()
    #print("Msg received:", repr(msg))

    parts = [p.strip() for p in msg.split(",") if p.strip()]
    for token in parts:
        if token.lower() == "end":
            print("End of values")
            done.set()
            return

        try:
            time_str, pitch_str, roll_str, yaw_str = token.split(":", 3)
            imu_values.append([float(pitch_str), float(roll_str), float(yaw_str)])
            time_values.append(int(time_str))
        except ValueError:
            print("Bad token:", repr(token))

def get_imu_data(timeout_s=40.0):
    time_values.clear()
    imu_values.clear()
    done.clear()
    
    # send command (NO await)
    ble.send_command(CMD.GET_IMU_READINGS, "")

    t0 = time.time()
    while not done.is_set() and (time.time() - t0) < timeout_s:
        ble.sleep(0.01)

    if not done.is_set():
        print("Timeout reached.")
    return list(imu_values), list(time_values)



# start notify
try:
    connectBLE(ble)
    
    ble.start_notify(ble.uuid['RX_STRING'], imu_array_notification_callback)
except Exception as e:
    if "Notify acquired" in str(e):
        print("Notify already active; continuing.")
    else:
        raise
time.sleep(0.2)

imu_vals, time_vals = get_imu_data()

print("IMU array length: ", len(imu_values))
print("Time array length: ", len(time_values))
#print("IMU values: ",imu_vals)
#print("Time values:", time_vals)

t = np.array(time_vals, dtype=np.int64)

# if time_vals are in milliseconds:
dt = np.diff(t)              # dt in ms
ave_dt = dt.mean()           # average dt in ms
sample_rate_hz = 1000.0 / ave_dt
recording_time = (time_vals[-1] - time_vals[0]) / 1000 # Convert from ms to s

print(f"Average dt: {ave_dt:.2f} ms")
print("Recording time: ", recording_time, "s")

try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception as e:
    print("Failed to stop notifications with exception: ", e)


End of values
IMU array length:  2000
Time array length:  2000
IMU values:  [[17.020918, 7.524826, 65.5], [16.962591, 7.619891, 65.5], [16.933908, 7.62306, 65.5], [16.910597, 7.579535, 65.5], [16.863403, 7.568234, 65.5], [16.895365, 7.498307, 65.5], [16.875704, 7.475224, 65.5], [16.945126, 7.438339, 65.5], [16.916613, 7.376732, 65.5], [16.907564, 7.394779, 65.5], [16.939358, 7.419626, 65.4], [16.90859, 7.425251, 65.4], [16.924208, 7.522503, 65.4], [16.908302, 7.470135, 65.4], [16.938419, 7.481975, 65.4], [16.890015, 7.540468, 65.5], [16.880657, 7.510424, 65.5], [16.917051, 7.480068, 65.5], [16.938843, 7.473276, 65.5], [16.97716, 7.485725, 65.5], [16.989489, 7.544491, 65.4], [16.988499, 7.527567, 65.4], [17.002905, 7.543525, 65.4], [16.970945, 7.634994, 65.5], [16.9781, 7.615246, 65.5], [16.958876, 7.636292, 65.5], [16.947138, 7.686365, 65.5], [16.901983, 7.609693, 65.5], [16.851767, 7.55054, 65.5], [16.895823, 7.508985, 65.5], [16.892368, 7.540431, 65.5], [16.871029, 7.546222, 65.5], [

### Start recording

In [48]:
# Send the message

try:
    connectBLE(ble)
except Exception as e:
    print("Exception")

ble.send_command(CMD.START_RECORDING, "")

# Check that the board received it and returns it
s = ble.receive_string(ble.uuid['RX_STRING'])
print("Recieved from board: " + s)

2026-02-10 20:38:57,354 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:42:d1:79:ab:49
2026-02-10 20:38:57,354 | INFO     |: Scanning for device with address: c0:42:d1:79:ab:49, service UUID: 4843fd62-068c-4a37-af34-e724c6a05681
2026-02-10 20:39:07,471 | INFO     |: Found 20 total devices
2026-02-10 20:39:07,472 | INFO     |: Found matching device: C0:42:D1:79:AB:49 (name: Artemis BLE)
2026-02-10 20:39:08,844 | INFO     |: Connected to c0:42:d1:79:ab:49
Recieved from board: Started recording


### Stop Recording

In [19]:
# Send the message

try:
    connectBLE(ble)
except Exception as e:
    print("Exception")

ble.send_command(CMD.STOP_RECORDING, "")

# Check that the board received it and returns it
s = ble.receive_string(ble.uuid['RX_STRING'])
print("Recieved from board: " + s)

Recieved from board: Stopped recording
